# Relatório da Fase 1 - Modelagem de Tópicos Concluída
Neste caderno documentamos a estratégia bem-sucedida de substituição das regras de palavras-chave do ludo-prism por um motor de agrupamento inteligente (BERTopic).

## Nossas Conquistas:
1. **Pipeline Duplo Funcional**: Dividindo o corpus previamente em grupos de "Sentimentos", nossa IA agora atrela emoções aos clusters ao invés de atrelar assuntos indiferentes.
2. **Supressão do Modelo Baseado Em Lógica LNI**: Cancelamos a integração que exigia usar *mDeBERTa-v3* como força motriz Zero-Shot. Concluímos que forçar categorias inibia o modelo de descobrir sub-nichos altamente relevantes (Como clusters separados reclamando de bola/cross do FIFA e outro só reclamando das marchas de jogos de corrida).
3. **Evasão Total do Ruído Espacial (Tópico -1)**: O algoritmo tradicional HDBSCAN costuma falhar por exigir similaridade idêntica entre os elementos. Tornando o agrupador (UMAP/HDBSCAN) menos ríspido, e depois distribuindo as orfãs na força por distânciamento numérico `topic_model.reduce_outliers(..., strategy="embeddings")`, zeramos a negação e aumentamos a população em amostras super específicas, atingindo nosso objetivo de captar gírias regionais e dores cruciais na análise de sentimentos.

# 1. Configuração e Carregamento de Dados
Vamos carregar o dataset que preparamos com os sentimentos extraídos e instalar as bibliotecas necessárias para a modelagem de tópicos.

In [1]:
!pip install pandas bertopic fastparquet

import pandas as pd
from bertopic import BERTopic

# Caminho para o arquivo gerado na Fase 1
caminho_arquivo = "data/processed/steam_reviews_sentences_COM_SENTIMENTO.parquet"

try:
    df = pd.read_parquet(caminho_arquivo)
    print(f"Dataset carregado com {len(df)} sentenças.")
    display(df.head())
except FileNotFoundError:
    print(f"Arquivo não encontrado: {caminho_arquivo}")



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
d:\workspace\ludo-prism\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset carregado com 190686 sentenças.


,review,voted_up,votes_up,votes_funny,author.playtime_at_review,language,genre,review_clean,word_count,sentence,sentiment_label
0,"Em minha opnião, o melhor post-apocalyptic da ...",True,5,0,2248.0,brazilian,Action,"Em minha opnião, o melhor postapocalyptic da a...",67,"Em minha opnião, o melhor postapocalyptic da a...",POS
1,"Joguei mais um bocado do Evolve, e até termine...",False,6,1,139.0,brazilian,Action,"Joguei mais um bocado do Evolve, e até termine...",367,"Joguei mais um bocado do Evolve, e até termine...",NEU
2,"Joguei mais um bocado do Evolve, e até termine...",False,6,1,139.0,brazilian,Action,"Joguei mais um bocado do Evolve, e até termine...",367,"Primeira coisa, é que é um jogo muito, muito m...",NEG
3,"Joguei mais um bocado do Evolve, e até termine...",False,6,1,139.0,brazilian,Action,"Joguei mais um bocado do Evolve, e até termine...",367,Então se prepare para ficar puto jogando esse ...,NEG
4,"Joguei mais um bocado do Evolve, e até termine...",False,6,1,139.0,brazilian,Action,"Joguei mais um bocado do Evolve, e até termine...",367,"Segunda coisa, esse jogo é muito desbalanceado...",NEG


# 2. Separação por Sentimento
De acordo com o nosso plano (Pipeline Duplo), precisamos aplicar a modelagem de tópicos _dentro_ de cada classe de sentimento. Assim, evitamos que o modelo misture críticas e elogios num mesmo tópico (ex: "Bugs" vira "Bugs - Positivo" e "Bugs - Negativo" no conceito ideal, mas separando na base é melhor).

In [2]:
# Vamos visualizar a distribuição dos sentimentos
if 'sentiment_label' in df.columns: 
    print(df['sentiment_label'].value_counts())
    
    # Separando os dataframes
    df_pos = df[df['sentiment_label'] == 'POS']
    df_neg = df[df['sentiment_label'] == 'NEG']
    df_neu = df[df['sentiment_label'] == 'NEU']
    
    print("\nSeparação concluída!")
else:
    print("Por favor, verifique o nome da coluna de sentimento no DataFrame impresso acima e ajuste este código.")

sentiment_label
NEU    105132
POS     54995
NEG     30559
Name: count, dtype: int64

Separação concluída!


# 3. Modelagem de Tópicos (Pipeline Otimizado em Lote)
Agora vamos processar tanto a base de sentenças **Negativas** (críticas) quanto a de **Positivas** (elogios).
Encapsulamos a lógica validada anteriormente (UMAP permissivo + HDBSCAN + resgate agressivo por Embeddings) em uma função que vai varrer as duas emoções e armazenar o "cérebro" de cada uma separadamente na memória do notebook.

In [3]:
import os
import nltk
import random
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# Preparo das stopwords para o projeto
nltk.download('stopwords', quiet=True)
stop_words_pt = nltk.corpus.stopwords.words('portuguese')
stop_words_custom = stop_words_pt + ['jogo', 'jogos', 'pra', 'pro', 'ta', 'sobre', 'tudo', 'porque', 'então', 'entao', 'jogar', 'ser', 'vai']

def treinar_otimizar_bertopic(docs, label):
    print(f"\n========================================================")
    print(f"[{label}] Iniciando pipeline BERTopic para {len(docs)} sentenças...")
    print(f"========================================================")
    
    vectorizer_model = CountVectorizer(stop_words=stop_words_custom)
    
    # Parâmetros validados na primeira iteracão para evitar perda massiva de dados
    hdbscan_model = HDBSCAN(min_cluster_size=10, min_samples=5, prediction_data=True)
    umap_model = UMAP(n_neighbors=20, min_dist=0.0, random_state=42)
    
    topic_model = BERTopic(
        language="multilingual", 
        # CULPADO DO TRAVAMENTO CORRIGIDO: 
        # Modificado de True para False. Como estamos usando strategy="embeddings" na redução 
        # de outliers, não precisamos mais calcular a probabilidade matemática total.
        calculate_probabilities=False, 
        verbose=True,
        vectorizer_model=vectorizer_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model
    )
    
    print(f"[{label}] 1/3 - Treinando modelo e gerando clusters base...")
    topics, probs = topic_model.fit_transform(docs)
    
    freq_antes = topic_model.get_topic_info()
    ruido_antes = freq_antes[freq_antes['Topic'] == -1]['Count'].values[0] if -1 in freq_antes['Topic'].values else 0
    print(f"[{label}] Ruído antes da otimização: {ruido_antes} ({(ruido_antes/len(docs))*100:.2f}%)")
    
    print(f"[{label}] 2/3 - Aplicando reduce_outliers agressivo por embeddings...")
    new_topics = topic_model.reduce_outliers(
        docs, 
        topics, 
        strategy="embeddings", 
        threshold=0.15
    )
    topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer_model)
    
    freq_depois = topic_model.get_topic_info()
    ruido_depois = freq_depois[freq_depois['Topic'] == -1]['Count'].values[0] if -1 in freq_depois['Topic'].values else 0
    print(f"[{label}] Ruído após otimização (Alvo ~ ZERO): {ruido_depois} ({(ruido_depois/len(docs))*100:.2f}%)")
    
    # Gerar o dataset consolidado dos tópicos
    df_docs_topics = pd.DataFrame({'sentence': docs, 'topic': new_topics})
    
    print(f"[{label}] 3/3 - Pipeline finalizada com sucesso!\n")
    return topic_model, df_docs_topics, freq_depois

# 1. Obter listas limpas de documentos (sem valores nulos)
docs_neg = df_neg['sentence'].dropna().tolist()
docs_pos = df_pos['sentence'].dropna().tolist()

# 2. Objetos de estado para guardar resultados
modelos = {}
dataframes = {}
frequencias = {}

# 3. Execução sequencial para cada grupo de sentimento
modelos['NEG'], dataframes['NEG'], frequencias['NEG'] = treinar_otimizar_bertopic(docs_neg, 'NEG')
modelos['POS'], dataframes['POS'], frequencias['POS'] = treinar_otimizar_bertopic(docs_pos, 'POS')

2026-05-19 18:03:34,702 - BERTopic - Embedding - Transforming documents to embeddings.



[NEG] Iniciando pipeline BERTopic para 30559 sentenças...
[NEG] 1/3 - Treinando modelo e gerando clusters base...


Batches: 100%|██████████| 955/955 [05:01<00:00,  3.16it/s]
2026-05-19 18:08:42,169 - BERTopic - Embedding - Completed ✓
2026-05-19 18:08:42,170 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-19 18:09:25,074 - BERTopic - Dimensionality - Completed ✓
2026-05-19 18:09:25,075 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-19 18:09:27,805 - BERTopic - Cluster - Completed ✓
2026-05-19 18:09:27,812 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-19 18:09:28,338 - BERTopic - Representation - Completed ✓


[NEG] Ruído antes da otimização: 11044 (36.14%)
[NEG] 2/3 - Aplicando reduce_outliers agressivo por embeddings...


2026-05-19 18:11:37,246 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-05-19 18:11:38,755 - BERTopic - Embedding - Transforming documents to embeddings.


[NEG] Ruído após otimização (Alvo ~ ZERO): 0 (0.00%)
[NEG] 3/3 - Pipeline finalizada com sucesso!


[POS] Iniciando pipeline BERTopic para 54995 sentenças...
[POS] 1/3 - Treinando modelo e gerando clusters base...


Batches: 100%|██████████| 1719/1719 [08:13<00:00,  3.48it/s]
2026-05-19 18:19:58,976 - BERTopic - Embedding - Completed ✓
2026-05-19 18:19:58,977 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-19 18:20:47,587 - BERTopic - Dimensionality - Completed ✓
2026-05-19 18:20:47,589 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-19 18:20:51,222 - BERTopic - Cluster - Completed ✓
2026-05-19 18:20:51,235 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-19 18:20:52,106 - BERTopic - Representation - Completed ✓


[POS] Ruído antes da otimização: 18592 (33.81%)
[POS] 2/3 - Aplicando reduce_outliers agressivo por embeddings...


2026-05-19 18:23:51,019 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


[POS] Ruído após otimização (Alvo ~ ZERO): 0 (0.00%)
[POS] 3/3 - Pipeline finalizada com sucesso!



In [4]:
# Ver visualmente quais Tópicos foram estruturados para ambas as classes
print("--- 15 PRINCIPAIS TÓPICOS: NEGATIVOS ---")
display(frequencias['NEG'][['Topic', 'Count', 'Name']].head(15))

print("\n--- 15 PRINCIPAIS TÓPICOS: POSITIVOS ---")
display(frequencias['POS'][['Topic', 'Count', 'Name']].head(15))

--- 15 PRINCIPAIS TÓPICOS: NEGATIVOS ---


,Topic,Count,Name
0,0,634,0_joguei_fiquei_gostei_consegui
1,1,565,1_polenguinho_nada_zero_nenhuma
2,2,417,2_pc_ram_fps_rodar
3,3,332,3_trilha_sonora_música_músicas
4,4,301,4_preço_vale_reais_pagar
5,5,261,5_português_tradução_inglês_portugues
6,6,278,6_preço_vale_caro_promoção
7,7,255,7_carro_carros_volante_moto
8,8,258,8_armas_inimigos_arma_inimigo
9,9,195,9_steam_origin_versão_epic



--- 15 PRINCIPAIS TÓPICOS: POSITIVOS ---


,Topic,Count,Name
0,0,974,0_amigos_sozinho_amigo_risadas
1,1,783,1_gráficos_gráfico_graficos_bonitos
2,2,724,2_gráficos_bonitos_gráfico_lindos
3,3,409,3_português_brasileiro_brasileiros_brasil
4,4,346,4_steam_oficina_versão_comprei
5,5,425,5_sonora_trilha_soundtrack_combina
6,6,352,6_montaria_porra_lagrimas_arrancar
7,7,340,7_bugs_atualizações_atualização_patch
8,8,300,8_sonoros_som_efeitos_sonora
9,9,304,9_campanha_arkham_campanhas_batman


# 4. Exportação Persistente (Para o LudoPrism)
Salvaremos os modelos validados no disco rígido. 
Amanhã, o seu sistema web (Backend) só precisará usar uma simples função `BERTopic.load('dir')` nessa mesma pasta para carregar o cérebro sem precisar retreinar nada. O novo texto só precisa rodar o `.transform()`.

In [5]:
# Criar pasta estática para exportação dos pesos do ML
pasta_saida = "data/models/bertopic"
os.makedirs(pasta_saida, exist_ok=True)

print(">>> Salvando modelo estrutural BERTopic NEGATIVO...")
# save_ctfidf=True para extração granular das representacoes
modelos['NEG'].save(f"{pasta_saida}/ludoprism_negativo_dir", serialization="safetensors", save_ctfidf=True)

print(">>> Salvando modelo estrutural BERTopic POSITIVO...")
modelos['POS'].save(f"{pasta_saida}/ludoprism_positivo_dir", serialization="safetensors", save_ctfidf=True)

# Exportar também um CSV de segurança com as classificações finais pra você visualizar nas tabelas se precisar
dataframes['NEG'].to_csv(f"{pasta_saida}/cluster_negativo_auditoria.csv", index=False)
dataframes['POS'].to_csv(f"{pasta_saida}/cluster_positivo_auditoria.csv", index=False)

print("\n✅ Vencedores Cimentados! Os Modelos foram armazenados permanentemente em 'data/models/bertopic/'.")

>>> Salvando modelo estrutural BERTopic NEGATIVO...
>>> Salvando modelo estrutural BERTopic POSITIVO...

✅ Vencedores Cimentados! Os Modelos foram armazenados permanentemente em 'data/models/bertopic/'.


In [7]:
# 5. Auditoria de Leitura Viva
# Aqui você pode sortear amostras rápidas mudando alí de POS para NEG pra ter certeza do aprendizado

def exibir_amostra_topico_dinamica(label):
    freq_df = frequencias[label]
    df_top = dataframes[label]
    
    top_topics = freq_df[freq_df['Topic'] != -1].head(10)
    
    print(f"========================================================")
    print(f"  AUDITORIA ALEATÓRIA: SENTIMENTO {label} ")
    print(f"========================================================")
    
    for _, row in top_topics.iterrows():
        topic_id = row['Topic']
        nome = row['Name']
        textos_topico = df_top[df_top['topic'] == topic_id]['sentence'].tolist()
        
        print(f"\n[TÓPICO {topic_id}] {nome.upper()}")
        
        k_amostras = min(3, len(textos_topico))
        amostra = random.sample(textos_topico, k=k_amostras)
        for i, texto in enumerate(amostra, 1):
            print(f"  {i}. {texto}")

# Deixei pré-configurado os elogios: mude pra 'NEG' se quiser revisar as reclamações
exibir_amostra_topico_dinamica('NEG')

  AUDITORIA ALEATÓRIA: SENTIMENTO NEG 

[TÓPICO 0] 0_JOGUEI_FIQUEI_GOSTEI_CONSEGUI
  1. Eu fui jogar pela primeira vez, e estava muito desbalanceado, eu um mero lvl 1 contra um level 150 e um lvl 40, eles ficaram me zoando e me bulinando quando eu era jason.
  2. Esse jogo foi muito frustrante pra mim.
  3. Não vou terminar o jogo, mas não é por medo ou por não saber o que fazer, mas sim porque o achei muito entediante e redundante.

[TÓPICO 1] 1_POLENGUINHO_NADA_ZERO_NENHUMA
  1. Mas nao entenda errado.
  2. Não há a menor necessidade de ser furtivo em Rage 2.
  3. Não achei o sv balanceado.

[TÓPICO 2] 2_PC_RAM_FPS_RODAR
  1. Como não obtive o reembolso troquei a minha placa de vídeo e mesmo assim o game da ERRO FATAL algumas vezes dizendo que tem arquivos corrompidos, eu já verifiquei a integridade dos arquivos desinstalei e instalei o jogo de novo e continua esse mesmo erro FATAL então não adiantou eu trocar a placa de video conseguir jogar por uma hora sem dar tela preta e o jogo 